# 04 — Export Data for Power BI
This notebook re-runs the key analysis and model steps, then exports clean CSV files into `dashboard/powerbi_data/` — ready to import directly into Power BI Desktop.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

os.makedirs('../dashboard/powerbi_data', exist_ok=True)

df = pd.read_csv('../data/processed/opd_visits_clean.csv')
df.shape

## 1. Export the main cleaned dataset (for slicers/filters in Power BI)

In [ ]:
df.to_csv('../dashboard/powerbi_data/opd_visits_clean.csv', index=False)
print('Exported opd_visits_clean.csv:', df.shape)

## 2. Export summary tables (pre-aggregated, for quick KPI cards & charts)
Includes a `DayOrder` helper column (Mon=1 ... Sun=7) so Power BI sorts DayOfWeek correctly instead of alphabetically.

In [ ]:
day_order_map = {'Mon': 1, 'Tue': 2, 'Wed': 3, 'Thu': 4, 'Fri': 5, 'Sat': 6, 'Sun': 7}

dept_summary = df.groupby('Department')['WaitTimeMinutes'].agg(
    avg_wait_minutes='mean', total_visits='count', max_wait_minutes='max'
).round(1).reset_index().sort_values('avg_wait_minutes', ascending=False)
dept_summary.to_csv('../dashboard/powerbi_data/department_summary.csv', index=False)

hourly_summary = df.groupby('ArrivalHour')['WaitTimeMinutes'].agg(
    avg_wait_minutes='mean', total_visits='count'
).round(1).reset_index().sort_values('ArrivalHour')
hourly_summary.to_csv('../dashboard/powerbi_data/hourly_summary.csv', index=False)

dow_summary = df.groupby('DayOfWeek')['WaitTimeMinutes'].agg(
    avg_wait_minutes='mean', total_visits='count'
).round(1).reset_index()
dow_summary['DayOrder'] = dow_summary['DayOfWeek'].map(day_order_map)
dow_summary = dow_summary.sort_values('DayOrder')
dow_summary.to_csv('../dashboard/powerbi_data/dayofweek_summary.csv', index=False)

triage_summary = df.groupby('TriageCategory')['WaitTimeMinutes'].agg(
    avg_wait_minutes='mean', total_visits='count'
).round(1).reset_index().sort_values('avg_wait_minutes', ascending=False)
triage_summary.to_csv('../dashboard/powerbi_data/triage_summary.csv', index=False)

weekend_summary = df.groupby('IsWeekend')['WaitTimeMinutes'].agg(
    avg_wait_minutes='mean', total_visits='count'
).round(1).reset_index()
weekend_summary.to_csv('../dashboard/powerbi_data/weekend_summary.csv', index=False)

print('Exported 5 summary tables.')
dept_summary

## 3. Export Age Group summary (for donut/bar chart)

In [ ]:
agegroup_summary = df.groupby('AgeGroup')['WaitTimeMinutes'].agg(
    avg_wait_minutes='mean', total_visits='count'
).round(1).reset_index().sort_values('total_visits', ascending=False)
agegroup_summary.to_csv('../dashboard/powerbi_data/agegroup_summary.csv', index=False)
agegroup_summary

## 4. Export Day × Hour heatmap data (with DayOrder for correct sort)

In [ ]:
heatmap_data = df.groupby(['DayOfWeek', 'ArrivalHour'])['WaitTimeMinutes'].mean().round(1).reset_index()
heatmap_data.columns = ['DayOfWeek', 'ArrivalHour', 'avg_wait_minutes']
heatmap_data['DayOrder'] = heatmap_data['DayOfWeek'].map(day_order_map)
heatmap_data = heatmap_data.sort_values(['DayOrder', 'ArrivalHour'])
heatmap_data.to_csv('../dashboard/powerbi_data/heatmap_data.csv', index=False)
print('Exported heatmap_data.csv:', heatmap_data.shape)
heatmap_data.head(10)

## 5. Export overall KPIs (single-row table for KPI cards)

In [ ]:
kpi = pd.DataFrame([{
    'total_patients': len(df),
    'avg_wait_minutes': round(df['WaitTimeMinutes'].mean(), 1),
    'max_wait_minutes': df['WaitTimeMinutes'].max(),
    'min_wait_minutes': df['WaitTimeMinutes'].min(),
    'avg_total_time_in_hospital': round(df['TotalTimeInHospital'].mean(), 1),
}])
kpi.to_csv('../dashboard/powerbi_data/overall_kpis.csv', index=False)
kpi

## 6. Train and compare 3 models: Linear Regression, KNN, Random Forest
Same models and tuning as `02_wait_time_prediction.ipynb` — kept in sync so the Model Performance page in Power BI matches the notebook results exactly.

In [ ]:
categorical_features = ['Department', 'DayOfWeek', 'TriageCategory', 'AppointmentType']
numeric_features = ['ArrivalHour', 'IsWeekend', 'FacilityOccupancyRate', 'ProvidersOnShift', 'NursesOnShift', 'StaffToPatientRatio']
target = 'WaitTimeMinutes'

X = df[categorical_features + numeric_features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)],
    remainder='passthrough'
)

def accuracy_within(y_true, y_pred, threshold=15):
    return np.mean(np.abs(y_true.values - y_pred) <= threshold) * 100

In [ ]:
# Linear Regression (baseline)
lr_model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

# KNN (tuned)
knn_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', KNeighborsRegressor())])
knn_grid = {
    'regressor__n_neighbors': [5, 10, 15, 20, 30, 50],
    'regressor__weights': ['uniform', 'distance'],
}
knn_search = GridSearchCV(knn_pipeline, knn_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
knn_search.fit(X_train, y_train)
knn_model = knn_search.best_estimator_
knn_preds = knn_model.predict(X_test)
print('Best KNN params:', knn_search.best_params_)

# Random Forest (tuned)
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(random_state=42))])
rf_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [4, 5, 6, 8, 12],
    'regressor__min_samples_leaf': [1, 5, 10],
}
rf_search = GridSearchCV(rf_pipeline, rf_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
rf_search.fit(X_train, y_train)
rf_model = rf_search.best_estimator_
rf_preds = rf_model.predict(X_test)
print('Best Random Forest params:', rf_search.best_params_)

## 7. Export model comparison metrics

In [ ]:
results = []
for name, preds in [('Linear Regression', lr_preds), ('KNN', knn_preds), ('Random Forest', rf_preds)]:
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    acc = accuracy_within(y_test, preds)
    results.append({'Model': name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R2': round(r2, 3), 'Accuracy_pct': round(acc, 1)})

results_df = pd.DataFrame(results).sort_values('MAE')
results_df.to_csv('../dashboard/powerbi_data/model_comparison.csv', index=False)
results_df

## 8. Export actual vs. predicted sample (for the line chart in Power BI)

In [ ]:
predictions = {'Actual': y_test.reset_index(drop=True)}
predictions['Linear Regression'] = pd.Series(lr_preds)
predictions['KNN'] = pd.Series(knn_preds)
predictions['Random Forest'] = pd.Series(rf_preds)

pred_df = pd.DataFrame(predictions)
pred_df = pred_df.sort_values('Actual').reset_index(drop=True).head(60)
pred_df['PatientIndex'] = range(1, len(pred_df) + 1)
pred_df.to_csv('../dashboard/powerbi_data/actual_vs_predicted.csv', index=False)
pred_df.head()

In [ ]:
print('All Power BI data files exported to dashboard/powerbi_data/:')
for f in sorted(os.listdir('../dashboard/powerbi_data')):
    print(' -', f)